# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ali0678/FlyRank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import files

uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("content_refresh_anonymized.csv")

lane_df = (
    df[
        (df["impressions_90d"] > 0) &
        (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

print("Rows:", len(lane_df))
print("Unique content IDs:", lane_df["content_id"].nunique())

Rows: 30000
Unique content IDs: 30000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule prioritizes pages that are both old and have meaningful search volume.

The idea is that an older page with substantial impressions has more potential value from a refresh than a new page or a page with very little visibility.

Rule: Give higher scores to pages that are older and have more impressions.

Reason code: stale_high_volume

Action: refresh_review

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
age_bins = pd.cut(
    lane_df["content_age_days"],
    bins=[89, 180, 365, 730, np.inf],
    labels=["90-180d", "181-365d", "366-730d", "730d+"]
)

age_check = (
    lane_df.assign(age_bucket=age_bins)
    .groupby("age_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        avg_impressions=("impressions_90d", "mean"),
        pct_declining=("trend_direction", lambda x: (x == "down").mean())
    )
    .reset_index()
)

display(age_check)

print("VERDICT: MIXED")

volume_bins = pd.qcut(
    lane_df["impressions_90d"],
    q=4,
    duplicates="drop"
)

volume_check = (
    lane_df.assign(volume_bucket=volume_bins)
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id", "size"),
        avg_age_days=("content_age_days", "mean"),
        pct_declining=("trend_direction", lambda x: (x == "down").mean())
    )
    .reset_index()
)

display(volume_check)

print("VERDICT: CONFIRMED")


,age_bucket,n,avg_impressions,pct_declining
0,90-180d,12272,5025.862451,0.627282
1,181-365d,11368,5398.772871,0.514866
2,366-730d,6360,5182.445755,0.426258


VERDICT: MIXED


,volume_bucket,n,avg_age_days,pct_declining
0,"(0.999, 81.0]",7503,241.600160,0.376116
1,"(81.0, 731.0]",7499,266.988665,0.604614
2,"(731.0, 3615.25]",7498,264.400107,0.625634
3,"(3615.25, 517715.0]",7500,251.691733,0.562000


VERDICT: CONFIRMED


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

I will use one simple baseline score combining staleness and volume. Older pages receive a higher staleness score, while pages with more impressions receive a higher volume score.

The action is `refresh_review` and the only reason code is `stale_high_volume`.

This is a baseline for comparison with the later ML model, not a claim that these pages will definitely improve after a refresh.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Make copies so the original data is unchanged
baseline = lane_df.copy()

# Staleness score: older pages get higher scores
baseline["staleness_score"] = (
    baseline["content_age_days"] / baseline["content_age_days"].max()
)

# Volume score: log reduces the effect of extremely large impression counts
baseline["volume_score"] = (
    np.log1p(baseline["impressions_90d"]) /
    np.log1p(baseline["impressions_90d"].max())
)

# One simple baseline score
baseline["score"] = (
    0.6 * baseline["staleness_score"] +
    0.4 * baseline["volume_score"]
)

baseline["reason_code"] = "stale_high_volume"
baseline["action"] = "refresh_review"

# Rank highest score first
baseline = baseline.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

output_cols = [
    "rank",
    "content_id",
    "score",
    "action",
    "reason_code",
    "content_age_days",
    "impressions_90d"
]

baseline_output = baseline[output_cols]

display(baseline_output.head(20))

import os

os.makedirs("work/outputs", exist_ok=True)

baseline_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:")
print("work/outputs/baseline_action_score.csv")

,rank,content_id,score,action,reason_code,content_age_days,impressions_90d
0,1,content_5fe46e04994d,0.971277,refresh_review,stale_high_volume,537,517715
1,2,content_9b934e3e7101,0.923170,refresh_review,stale_high_volume,537,106384
2,3,content_fca1bf3940c0,0.916764,refresh_review,stale_high_volume,537,86170
3,4,content_82572b951646,0.914753,refresh_review,stale_high_volume,537,80655
4,5,content_57971022aadc,0.914642,refresh_review,stale_high_volume,545,60739
5,6,content_8a8b6089b6da,0.908333,refresh_review,stale_high_volume,545,49356
6,7,content_6989c356365e,0.907462,refresh_review,stale_high_volume,545,47962
7,8,content_1a9e894be2e2,0.906129,refresh_review,stale_high_volume,482,416180
8,9,content_f516cec15df8,0.905147,refresh_review,stale_high_volume,545,44445
9,10,content_e28ccaa8e211,0.904003,refresh_review,stale_high_volume,537,56633


Saved:
work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 recommendations rather than assuming that a high score means the recommendation is automatically correct.

For each page, I record the action, the reason code, a confidence note, and what could make the recommendation wrong.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = baseline.head(20).copy()

top20["confidence_note"] = np.where(
    (top20["content_age_days"] >= 365) &
    (top20["impressions_90d"] >= 1000),
    "Strong baseline match",
    "Moderate baseline match"
)

top20["what_would_make_it_wrong"] = (
    "The page may not need a refresh despite being old and visible."
)

review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
1,2,content_9b934e3e7101,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
2,3,content_fca1bf3940c0,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
3,4,content_82572b951646,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
4,5,content_57971022aadc,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
5,6,content_8a8b6089b6da,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
6,7,content_6989c356365e,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
7,8,content_1a9e894be2e2,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
8,9,content_f516cec15df8,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...
9,10,content_e28ccaa8e211,refresh_review,stale_high_volume,Strong baseline match,The page may not need a refresh despite being ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some high-ranked pages may be weak recommendations because age and impressions alone do not tell us whether a refresh would actually improve performance. A page may be old and highly visible but already perform well or have no meaningful content problem.

I did not use existing FlyRank product flags, priority scores, or future outcomes as model inputs. The baseline uses only observable page-level signals available in the starter data.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Look at the bottom of the top 20 to find weaker picks
print("Lowest-scoring picks within the top 20:")
display(
    baseline.head(20).tail(5)[
        [
            "rank",
            "content_id",
            "score",
            "content_age_days",
            "impressions_90d",
            "action",
            "reason_code"
        ]
    ]
)

# Leakage check
possible_leakage = [
    "health_score",
    "priority_score",
    "action_type",
    "flag",
    "future_impressions",
    "future_sessions"
]

used_columns = set(baseline.columns)

print("\nLeakage check:")
for col in possible_leakage:
    if col in used_columns:
        print("CHECK:", col)
    else:
        print("OK:", col, "not used")

Lowest-scoring picks within the top 20:


,rank,content_id,score,content_age_days,impressions_90d,action,reason_code
15,16,content_1210e6c2e909,0.887858,537,33298,refresh_review,stale_high_volume
16,17,content_609902f4bc8c,0.886416,537,31756,refresh_review,stale_high_volume
17,18,content_e81b02320098,0.885861,537,31181,refresh_review,stale_high_volume
18,19,content_f04ac0af071c,0.885398,537,30710,refresh_review,stale_high_volume
19,20,content_da2adacc2e93,0.885073,537,30384,refresh_review,stale_high_volume



Leakage check:
OK: health_score not used
OK: priority_score not used
OK: action_type not used
OK: flag not used
OK: future_impressions not used
OK: future_sessions not used


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.